# 경구약제 Object Detection 실험

다양한 모델 비교 실험 노트북

**데이터 소스**
- AI Hub 경구약제 조합 데이터 (TS_01~08, TS_02 제외)
- 기존 Kaggle 대회 데이터 (`data/pill_yolo/`, 56클래스)

## 1. 데이터 준비

두 가지 데이터 소스 중 하나를 선택:
- **A. AI Hub 데이터**: 대량 데이터, 서브셋(tiny/small/medium/full) 지원
- **B. 기존 Kaggle 데이터**: `data/pill_yolo/`에 이미 준비된 56클래스 데이터

In [ ]:
from utils.sys_utils import add_experiment_root

PROJECT_ROOT = add_experiment_root(2)
PROJECT_ROOT

In [ ]:
from utils.date_uilts import get_current_date_for_filename

start_datetime = get_current_date_for_filename()

In [ ]:
from src.data.aihub_converter import convert_aihub_to_coco

# from src.data.yolo_to_coco import load_yolo_as_coco
from src.data.subset import create_subset  # , save_split
from src.data.coco_to_yolo import prepare_yolo_dataset

# from src.data.dataset import (
#     PillDetectionDataset,
#     collate_fn,
#     get_train_augment,
# )
# from src.models.builder import build_model
# from src.models.yolo_wrapper import train_yolo, evaluate_yolo
from src.train import train_torchvision
from src.train_yolo import run_yolo_experiment
from src.utils import load_config  # , set_seed, get_device

CONFIGS_DIR = PROJECT_ROOT / "configs"
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"
DATA_DIR = PROJECT_ROOT / "data"
KAGGLE_YOLO_DIR = PROJECT_ROOT / "data" / "pill_yolo"

In [ ]:
# ============================
# A. AI Hub 데이터 사용 시
# ============================
# combo_nums: 사용할 조합 번호 (2번은 Kaggle 테스트셋이므로 자동 차단)
coco = convert_aihub_to_coco(combo_nums=[1, 3, 4, 5, 6])

In [ ]:
# ============================
# C. Kaggle 대회 데이터 사용 시 (A/B 대신 이 셀 실행)
# ============================
# Kaggle train_images + train_annotations → COCO 변환 (56 클래스, 올바른 category_id)
# 제출 파일 생성 시 class_map.json이 자동으로 함께 저장됨
from src.data.kaggle_converter import convert_kaggle_to_coco

# coco = convert_kaggle_to_coco()
# TIER = "full"
# DATA_SOURCE = "kaggle"

In [ ]:
# 서브셋 생성: tiny(100장), small(500장), medium(2000장), full(전체)
TIER = "small"  # 변경하여 데이터 규모 조절
DATA_SOURCE = "aihub"

train_coco, val_coco = create_subset(coco, tier=TIER, seed=42)

### B. 기존 Kaggle 데이터 사용 시 (위 A 대신 이 셀 실행)

In [ ]:
# ============================
# B. 기존 Kaggle 대회 데이터 사용 시 (A 대신 이 셀 실행)
# ============================
# data/pill_yolo/ 의 YOLO 포맷 데이터를 COCO로 변환하여 torchvision 모델에서도 사용 가능
# train_coco = load_yolo_as_coco(KAGGLE_YOLO_DIR, split="train")
# val_coco = load_yolo_as_coco(KAGGLE_YOLO_DIR, split="val")
# TIER = "kaggle"
# DATA_SOURCE = "kaggle"

## 2. YOLO 모델 실험

In [ ]:
# # YOLO용 데이터셋 준비
# if DATA_SOURCE == "kaggle":
#     # 기존 Kaggle 데이터는 이미 YOLO 포맷 → 바로 사용
#     yaml_path = KAGGLE_YOLO_DIR / "data.yaml"
#     print(f"기존 Kaggle YOLO 데이터 사용: {yaml_path}")
# else:
#     # AI Hub 데이터 → YOLO 포맷 변환 (심볼릭 링크로 디스크 절약)
#     yolo_dir = DATA_DIR / f"yolo_{TIER}"
#     yaml_path = prepare_yolo_dataset(train_coco, val_coco, yolo_dir, symlink=True)

In [ ]:
# # YOLO 실험 config 로드 & 학습
# yolo_config = load_config(CONFIGS_DIR / "experiment" / "exp001_yolo11n_tiny.yaml")

# # 필요시 config 오버라이드
# # yolo_config["training"]["epochs"] = 10
# # yolo_config["model"]["name"] = "yolo11s"
# yolo_config["data"]["subset"] = "small"

# yolo_metrics = run_yolo_experiment(yolo_config, yaml_path, EXPERIMENTS_DIR)
# print(yolo_metrics)

## 3. torchvision 모델 실험 (SSD / Faster R-CNN / RetinaNet / FCOS)

In [ ]:
# SSD300 실험
ssd_config = load_config(CONFIGS_DIR / "experiment" / "exp002_ssd300_tiny.yaml")
ssd_metrics = train_torchvision(ssd_config, train_coco, val_coco, EXPERIMENTS_DIR)
print(ssd_metrics)

In [ ]:
# Faster R-CNN 실험
frcnn_config = load_config(CONFIGS_DIR / "experiment" / "exp003_fasterrcnn_tiny.yaml")
frcnn_metrics = train_torchvision(frcnn_config, train_coco, val_coco, EXPERIMENTS_DIR)
print(frcnn_metrics)

In [ ]:
# RetinaNet 실험
retina_config = load_config(CONFIGS_DIR / "experiment" / "exp004_retinanet_tiny.yaml")
retina_metrics = train_torchvision(retina_config, train_coco, val_coco, EXPERIMENTS_DIR)
print(retina_metrics)

In [ ]:
# FCOS 실험
fcos_config = load_config(CONFIGS_DIR / "experiment" / "exp005_fcos_tiny.yaml")
fcos_metrics = train_torchvision(fcos_config, train_coco, val_coco, EXPERIMENTS_DIR)
print(fcos_metrics)

## 4. 결과 비교

In [ ]:
import json
import pandas as pd

results = []
for exp_dir in sorted(EXPERIMENTS_DIR.iterdir()):
    metrics_file = exp_dir / "metrics.json"
    if metrics_file.exists():
        m = json.load(open(metrics_file))
        results.append({
            "experiment": exp_dir.name,
            "model": m.get("model", "unknown"),
            "mAP@75": m.get("map75") or m.get("final_map75", 0),
            "mAP@75:95": m.get("map75_95") or m.get("final_map75_95", 0),
            "epochs": m.get("epochs", 0),
        })

df = pd.DataFrame(results)
df = df.sort_values("mAP@75:95", ascending=False)
df

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Apple SD Gothic Neo"
plt.rcParams["axes.unicode_minus"] = False

if not df.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    x = range(len(df))
    width = 0.25
    ax.bar(x, df["mAP@75"], width, label="mAP@75")
    ax.bar([i + width for i in x], df["mAP@75:95"], width, label="mAP@75:95")
    ax.set_xticks(x)
    ax.set_xticklabels(df["model"], rotation=45, ha="right")
    ax.set_ylabel("mAP")
    ax.set_title(f"모델별 성능 비교 (tier: {TIER})")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5. 커스텀 실험

config를 직접 수정하거나 새로 만들어서 실험할 수 있습니다.

In [ ]:
# 예시: YOLO11s + small 데이터셋 + 20 epoch
# custom_config = {
#     "experiment": {"name": "exp010_yolo11s_small", "seed": 42},
#     "data": {"subset": "small", "test_size": 0.2, "img_size": 640},
#     "model": {"type": "yolo", "name": "yolo11s", "pretrained": True},
#     "training": {"epochs": 20, "batch_size": 16, "optimizer": "SGD", "lr": 0.01},
# }
#
# # 데이터 재준비 (tier 변경 시)
# train_coco_s, val_coco_s = create_subset(coco, tier="small", seed=42)
# yaml_s = prepare_yolo_dataset(train_coco_s, val_coco_s, DATA_DIR / "yolo_small")
# run_yolo_experiment(custom_config, yaml_s, EXPERIMENTS_DIR)